# SolarSpine R2B · Reproducción verificable de la entrega 2026

**Método congelado:** YOLO26n-seg (detector A3) + NativeROIRefiner U-Net (R2A), máscaras de **2048 × 2048**, COCO RLE comprimido. El pipeline ejecuta dos procesos independientes; el refinador utiliza únicamente predicciones nuevas del detector, nunca las máscaras TEST reales ni el CSV de referencia.

**Objetivo técnico:** recuperar las 180 imágenes oficiales, verificar los 7 módulos y los 2 checkpoints, ejecutar una inferencia completa y comparar los **SHA256** de los dos CSV generados con los previamente enviados. El notebook no entrena, no optimiza umbrales, no consulta etiquetas TEST y no envía nada a Kaggle.

**Resultado ya conocido, NO generado por este notebook:** puntaje público 0,29 (mostrado por Kaggle), PQ de desarrollo Fold4 0,34815. La coincidencia de hashes aquí comprueba *reproducibilidad*, no una nueva mejora de PQ ni un puntaje privado.

**Entorno científicamente aceptado:** Python 3.13.x, Tesla T4, PyTorch 2.11.0+cu130, torchvision 0.26.0+cu130, CUDA 13.0; dependencias restantes con versiones fijadas. Si Colab ofrece un entorno diferente, el notebook **se detiene** y genera diagnóstico en lugar de cambiar la receta sin validación.


### Ejecución oficial del jurado — clonación pública y anónima

El repositorio **debe ser público**. No se requieren tokens ni acceso al Google Drive del autor. Coloca los 180 JPEG oficiales TEST en `IMAGES` antes del preflight; el notebook no lee etiquetas.

Los pesos se recuperan mediante Git LFS y se validan por SHA256 antes de la inferencia. El resultado se considera reproducido solo si los dos CSV regenerados coinciden exactamente con sus hashes originales.

**Comprobación de acceso:** el cuaderno realiza `git clone` de forma anónima con `credential.helper=` y, a continuación, `git lfs pull`. Si el repositorio continúa privado, esa etapa debe fallar; no se necesita ningún token en la entrega final.


## 1 · Configuración y evidencia esperada


In [ ]:
from pathlib import Path
import csv, datetime, hashlib, importlib.metadata as md, json, os, platform, shutil, subprocess, sys, time, zipfile

# Directorio con EXACTAMENTE las 180 imágenes JPEG oficiales (sin etiquetas/GT).
# Si las tienes en otro lugar, modifica solo esta ruta; no cambies el contenido de los JPEG.
IMAGES = Path('/content/official_test/test_images')

REPO_DIR = Path('/content/solarspine-r2b-jury-repository')
OUTPUT_BASE = Path('/content/solarspine-r2b-jury-runs')
EXPECTED_IMAGES = 180
EXPECTED_SHA = {
  'detector.csv': '347307dc95727dbd9ea97d9d474c2125eb5bb14515e79c8bce563e51ead61a39',
  'submission_r2b.csv': 'c241bda04c1fefdab5fe1b6b6d90f4aeac23a02a2003f830c0b9b1b289d3294f',
}
EXPECTED_WEIGHTS = {
  'models/detector/best.pt': 'bbb3c55175f68f80c213ac9fd6d1a1ab5bd63993696b609348d36f0472515875',
  'models/refiner/best.pt': 'e4b9c3bacce816cd1d4dfb863622b93774743a6a401ae844088e33540882f397',
}

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for part in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(part)
    return digest.hexdigest()

def run_checked(cmd, *, cwd=None, label=None, logfile=None):
    label = label or str(cmd[0]); print('\n>>>', label, flush=True)
    print('COMMAND:', ' '.join(map(str, cmd)), flush=True)
    if logfile is None:
        subprocess.run([str(x) for x in cmd], check=True, cwd=cwd)
        return
    logfile = Path(logfile)
    logfile.parent.mkdir(parents=True, exist_ok=True)
    with logfile.open('a', encoding='utf-8') as handle:
        proc = subprocess.Popen([str(x) for x in cmd], cwd=cwd,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                encoding='utf-8', errors='replace', bufsize=1)
        try:
            for line in proc.stdout:
                print(line, end='', flush=True)
                handle.write(line)
        finally:
            proc.stdout.close()
        code = proc.wait()
    if code != 0:
        raise RuntimeError(f'{label} failed with exit code {code}. Log: {logfile}')

print('CONFIGURATION_READY', {'images': str(IMAGES), 'repo': str(REPO_DIR)})


## 2 · Clonación anónima de GitHub con LFS


In [ ]:
# SolarSpine R2B · versión de jurado: clonación ESTRICTAMENTE pública y anónima.
# No se utiliza ningún token de GitHub, repositorio privado o credencial del autor.
from pathlib import Path
import os, subprocess
OWNER = "topassky3"
REPOSITORY = "solarspine-r2b-final-2026"
REPO_URL = "https://" + "github.com/" + OWNER + "/" + REPOSITORY + ".git"
EXPECTED_COMMIT_PREFIX = ""  # Fijar SHA completo del commit final antes de publicar.

assert REPO_URL == "https://github.com/topassky3/solarspine-r2b-final-2026.git"
assert not any(x in REPO_URL for x in ("[", "]", "(", ")", "\\", " "))
# Colab puede traer Git, pero no Git LFS. Ambos son necesarios para descargar los .pt.
def ensure_git_lfs():
    current = subprocess.run(["git", "lfs", "version"], capture_output=True, text=True)
    if current.returncode == 0:
        print("GIT_LFS_AVAILABLE", current.stdout.strip())
        return
    print("INSTALLING_GIT_LFS_IN_COLAB")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "git-lfs"], check=True)
    installed = subprocess.run(["git", "lfs", "version"], capture_output=True, text=True)
    if installed.returncode != 0:
        raise RuntimeError("Git LFS no pudo instalarse. Verifica el entorno Colab.")
    print("GIT_LFS_READY", installed.stdout.strip())

ensure_git_lfs()

if REPO_DIR.exists():
    raise RuntimeError("La carpeta clonada ya existe. Inicia un entorno nuevo de Colab.")

# Asegurar un clone anónimo incluso si el usuario tiene credenciales configuradas.
public_env = os.environ.copy()
public_env.update({"GIT_TERMINAL_PROMPT": "0", "GCM_INTERACTIVE": "never", "GIT_LFS_SKIP_SMUDGE": "1"})
public_env.pop("GIT_ASKPASS", None)
public_env.pop("GH_TOKEN", None)
public_env.pop("GITHUB_TOKEN", None)
probe = subprocess.run(
    ["git", "-c", "credential.helper=", "ls-remote", "--exit-code", REPO_URL, "HEAD"],
    capture_output=True, text=True, env=public_env
)
if probe.returncode:
    raise RuntimeError(
        "PUBLIC_ACCESS_NOT_READY: no es posible clonar anónimamente. "
        "El propietario debe publicar el repositorio DESPUÉS de la revisión de licencias. "
        f"Detalle: {probe.stderr[-600:]}"
    )

# Git LFS smudge is disabled at clone to make its subsequent transfer explicit.
clone = subprocess.run(
    ["git", "-c", "credential.helper=", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
    check=True, env=public_env
)
lfs = subprocess.run(
    ["git", "-C", str(REPO_DIR), "-c", "credential.helper=", "lfs", "pull"],
    check=True, env=public_env
)
commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
if EXPECTED_COMMIT_PREFIX and not commit.startswith(EXPECTED_COMMIT_PREFIX):
    raise RuntimeError(f"Commit no esperado: {commit}")
remote = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True
).strip()
assert remote == REPO_URL
SOURCE_METHOD = "public_git_anonymous"
print("SOLARSPINE_PUBLIC_REPOSITORY_CLONED", commit)


## 3 · Preflight SHA256 y lista oficial de 180 imágenes


In [ ]:
# La integridad del código y los pesos se verifica ANTES de inferir.
run_checked([sys.executable, REPO_DIR / 'scripts/preflight_release.py','--strict'],label='Strict source and checkpoint SHA256 gate')
for rel, expected_hash in EXPECTED_WEIGHTS.items():
    got = sha256_file(REPO_DIR/rel)
    assert got == expected_hash, f'Invalid checkpoint at {rel}: {got}'

# Verificación de entradas: no se leerán anotaciones ni CSV de referencia.
import re
assert IMAGES.is_dir(), f'No existe la carpeta TEST: {IMAGES}'
jpegs = sorted(p for p in IMAGES.rglob('*') if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg'})
assert len(jpegs) == EXPECTED_IMAGES, f'Se requieren {EXPECTED_IMAGES} JPEG oficiales; hay {len(jpegs)}'
assert len({p.stem for p in jpegs}) == EXPECTED_IMAGES, 'Hay ID de imagen repetido'
assert all(re.fullmatch(r'\d{14}[BCLMTU]h', p.stem) for p in jpegs), 'Nombres de imágenes no oficiales'
print('SOURCE_WEIGHTS_AND_OFFICIAL_TEST_ROSTER_PASS',len(jpegs),'JPEG images')


## 4 · Runtime congelado en Tesla T4 (sin modificar el algoritmo)


In [ ]:
# No actualizar Torch/CUDA automáticamente: preservar el runtime probado.
REQUIRED_PACKAGES = {
 'torch':'2.11.0+cu130','torchvision':'0.26.0+cu130',
 'numpy':'2.1.3','Pillow':'11.3.0','pycocotools':'2.0.11',
 'scipy':'1.16.3','opencv-python':'4.12.0.88','ultralytics':'8.4.177'
}
assert sys.version_info[:2] == (3,13), f'Python {sys.version.split()[0]}; el runtime validado usa 3.13.x'
import torch
assert torch.cuda.is_available(), 'Selecciona un runtime GPU en Colab: Runtime > Change runtime type > T4 GPU'
assert torch.cuda.get_device_capability(0) == (7,5) and 'T4' in torch.cuda.get_device_name(0), 'Se requiere Tesla T4'
assert torch.__version__ == REQUIRED_PACKAGES['torch'] and torch.version.cuda == '13.0', (
    f'Torch/CUDA incorrectos: {torch.__version__}, CUDA {torch.version.cuda}; NO sobrescribir el wheel sin validar'
)
import torchvision
assert torchvision.__version__ == REQUIRED_PACKAGES['torchvision'], f'Torchvision no coincide: {torchvision.__version__}'

# Heredar PyTorch CUDA compatible de Colab y poner SOLO el resto de versiones fijadas en venv.
VENV = Path('/content/solarspine-r2b-jury-venv')
PY = VENV / 'bin' / 'python'
if not PY.is_file():
    run_checked([sys.executable,'-m','venv','--system-site-packages','--without-pip',VENV],label='Create non-destructive venv')
# El pip anfitrión gestiona el entorno con --python; no ejecutar ensurepip.
pinned = [f'{name}=={version}' for name,version in REQUIRED_PACKAGES.items() if name not in {'torch','torchvision'}]
run_checked([sys.executable,'-m','pip','--python',PY,'install','--no-deps','--disable-pip-version-check',*pinned],
            label='Install pinned inference dependencies without replacing CUDA Torch')

runtime_script = """
import importlib.metadata as im, json, platform, sys, torch
expected = {
 'torch':'2.11.0+cu130','torchvision':'0.26.0+cu130','numpy':'2.1.3','Pillow':'11.3.0',
 'pycocotools':'2.0.11','scipy':'1.16.3','opencv-python':'4.12.0.88','ultralytics':'8.4.177'
}
versions = {name: im.version(name) for name in expected}
assert versions == expected, {'actual': versions, 'expected': expected}
assert torch.cuda.is_available() and torch.version.cuda == '13.0'
print('PINNED_RUNTIME_PASS',json.dumps({'python':platform.python_version(),'cuda':torch.version.cuda,
 'gpu':torch.cuda.get_device_name(0),'versions':versions},sort_keys=True))
"""
run_checked([PY,'-c',runtime_script],label='Check exact runtime inside venv')
print('INFERENCE_RUNTIME_READY')


## 5 · Inferencia GPU completa: YOLO26n-seg → U-Net


In [ ]:
# Ejecución REAL de las 180 imágenes; NO usar --smoke para esta evaluación.
# Si se corta Colab, la carpeta de salida se mantiene en el entorno solo durante la sesión.
# Los archivos diferentes están protegidos por run_inference.py.
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('jury_%Y%m%dT%H%M%SZ')
RUN_DIR = OUTPUT_BASE / RUN_ID
RUN_DIR.mkdir(parents=True,exist_ok=False)
DETECTOR = REPO_DIR/'models/detector/best.pt'
REFINER = REPO_DIR/'models/refiner'
cmd = [PY,REPO_DIR/'delivery/r2b/run_inference.py',
       '--images',IMAGES,'--detector-weights',DETECTOR,
       '--refiner-dir',REFINER,'--output-dir',RUN_DIR]
run_checked(cmd,label='Full 180-image GPU inference',logfile=RUN_DIR/'inference.log')
summary_path = RUN_DIR / 'summary.json'
assert summary_path.is_file(), 'La inferencia no completó summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
assert summary['complete'] is True and summary['smoke'] is False and summary['GT_used'] is False
assert summary['Kaggle_upload_executed'] is False
assert all(summary['stages'][step]['images'] == EXPECTED_IMAGES for step in ('detector','refiner'))
assert (RUN_DIR/'detector.csv').is_file() and (RUN_DIR/'submission_r2b.csv').is_file()
print('JURY_INFERENCE_FINISHED',RUN_DIR)


## 6 · Comprobación posterior de los dos CSV


In [ ]:
# Referencias SOLO en esta etapa posterior; no se entregan como entrada al ejecutor.
checks = {}
for name, expected_hash in EXPECTED_SHA.items():
    actual = sha256_file(RUN_DIR/name)
    checks[name] = {'sha256':actual,'expected_sha256':expected_hash,'exact':actual==expected_hash}
    print('CSV_SHA256',name,actual,'EXACT_MATCH' if actual==expected_hash else 'MISMATCH')

with (RUN_DIR/'submission_r2b.csv').open(newline='',encoding='utf-8') as f:
    reader = csv.DictReader(f)
    assert reader.fieldnames == ['filament_id','segmentation_rle'], 'Columnas Kaggle incorrectas'
    actual_rows = sum(1 for _ in reader)

manifests = {stage: summary['stages'][stage] for stage in ('detector','refiner')}
assert all(not manifests[s]['smoke'] and not manifests[s]['GT_used'] for s in manifests)
assert manifests['refiner']['predictions'] == actual_rows, 'No coincide el número de instancias'
assert actual_rows == 1422, 'La salida difiere en el número de instancias esperado para R2B'
assert len(manifests['refiner']['empty_image_ids']) == 6, 'El número de imágenes sin predicción difiere'
exact_match = all(rec['exact'] for rec in checks.values())
print('R2B_180_IMAGES_COMPLETE','rows',actual_rows,'empty',len(manifests['refiner']['empty_image_ids']))
print('REPRODUCIBILITY_RESULT', 'EXACT_MATCH' if exact_match else 'DIFFERENT_OUTPUT__INVESTIGATE')

EVIDENCE = {
    'status': 'EXACT_MATCH' if exact_match else 'DIFFERENT_OUTPUT__INVESTIGATE',
    'source_method': SOURCE_METHOD,
    'repo_commit': commit,
    'images': EXPECTED_IMAGES,
    'instances': actual_rows,
    'empty_images': len(manifests['refiner']['empty_image_ids']),
    'GT_read': False,
    'training_performed': False,
    'Kaggle_upload_performed': False,
    'source_and_weights_gate': 'PASS',
    'checks': checks,
    'runtime_requirement': 'Python 3.13 + Tesla T4 + CUDA13 + exact locked packages',
    'run_sha256': summary['run_sha256'],
    'run_complete': True,
    'note': 'Exact bytes demonstrate reproduction, not a new public/private PQ score.',
}
(RUN_DIR/'jury_reproduction_evidence.json').write_text(
    json.dumps(EVIDENCE,indent=2,sort_keys=True,ensure_ascii=False)+'\n',encoding='utf-8')
assert exact_match, 'Los CSV NO coinciden; conservar esta salida y el log, no afirmar éxito.'
print('SOLARSPINE_R2B_JURY_CSV_EXACT_MATCH')


## 7 · Evidencia y exportación sin datos privados


In [ ]:
# Paquete DE EVIDENCIA para el jurado/propietario: solo CSV regenerados, hashes y logs.
# Nunca incluimos JPEG TEST, pesos, GT ni credenciales.
ARCHIVE = RUN_DIR.parent / f'{RUN_DIR.name}_evidence.zip'
include = ['contract.json','detector.csv','submission_r2b.csv','summary.json',
           'jury_reproduction_evidence.json','inference.log',
           'detector/manifest.json','refiner/manifest.json',
           'detector/runtime.json','refiner/runtime.json']
with zipfile.ZipFile(ARCHIVE,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=7) as z:
    for rel in include:
        src=RUN_DIR/rel
        assert src.is_file(), f'No se encontró archivo de evidencia: {rel}'
        z.write(src,arcname=rel)
print('EVIDENCE_BUNDLE',str(ARCHIVE),'SHA256',sha256_file(ARCHIVE))
print('FINAL_SUBMISSION_CSV',str(RUN_DIR/'submission_r2b.csv'))
try:
    from IPython.display import display, FileLink
    display(FileLink(str(ARCHIVE)))
except ImportError:
    pass
print('READY_FOR_MANUAL_REVIEW__NO_AUTOMATIC_KAGGLE_OR_GOOGLE_FORM_SUBMISSION')


## Criterio de aceptación

La reproducción de esta nueva versión se considera **demostrada** solamente si la celda de comparación imprime `SOLARSPINE_R2B_JURY_CSV_EXACT_MATCH` después de ejecutar las 180 imágenes. Si falla cualquier condición, conserva `inference.log`, los dos manifiestos y `jury_reproduction_evidence.json` (si existe), y **no** sustituyas el CSV de Kaggle ni afirmes un nuevo resultado.

Un `EXACT_MATCH` no significa que haya aumentado el *Public Score* 0,29, ni que se haya efectuado la entrega final del formulario. La publicación del repositorio y del informe PDF requiere revisión de licencias, acceso anónimo y autorización del propietario.
